In [ ]:
%pip install ucimlrepo

### Import

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ucimlrepo import fetch_ucirepo

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    GridSearchCV
)

from sklearn.naive_bayes import GaussianNB
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

In [ ]:
data = fetch_ucirepo(id= 17)

In [ ]:
X= data.data.features
y= data.data.targets

In [ ]:
df = pd.concat([X, y], axis= 1)

In [ ]:
print("Dataset shape: ", df.shape)

In [ ]:
df

In [ ]:
sns.countplot(data=df, x="Diagnosis")
plt.title("Distribution of Benign and Malignant Samples")
plt.xlabel("Diagnosis")
plt.ylabel("Number of Samples")
plt.show()

In [ ]:
print("Dataset information:")
X.info()

print("\nMissing values:")
print(X.isnull().sum())

print("\nDuplicate feature rows:")
print(X.duplicated().sum())

print("\nDescriptive statistics:")
display(X.describe().T)

### Train Test Split

In [ ]:
y = (y["Diagnosis"] == "M").astype(int)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

In [ ]:
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

In [ ]:
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

In [ ]:
## Model with all the 30 features
model_30 = GaussianNB()

scores_30 = cross_validate(
    model_30,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring
)

results_30 = pd.DataFrame({
    metric: scores_30[f"test_{metric}"]
    for metric in scoring
})

print("Model 1: 30 Features")
display(results_30.round(4))

print("\nMean CV Scores:")
print(results_30.mean().round(4))

In [ ]:
mean_features = [
    "radius1",
    "texture1",
    "perimeter1",
    "area1",
    "smoothness1",
    "compactness1",
    "concavity1",
    "concave_points1",
    "symmetry1",
    "fractal_dimension1"
]

X_train_10 = X_train[mean_features]
X_test_10 = X_test[mean_features]

model_10 = GaussianNB()

scores_10 = cross_validate(
    model_10,
    X_train_10,
    y_train,
    cv=cv,
    scoring=scoring
)

results_10 = pd.DataFrame({
    metric: scores_10[f"test_{metric}"]
    for metric in scoring
})


print("\nMean CV Scores:")
print(results_10.mean().round(4))

In [ ]:
pipeline = Pipeline([
    ("selection", SelectKBest(score_func=f_classif)),
    ("model", GaussianNB())
])

param_grid = {
    "selection__k": [5, 10, 15, 20, 25, 30],
    "model__var_smoothing": [1e-10, 1e-9, 1e-8, 1e-7]
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring={
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1",
        "roc_auc": "roc_auc"
    },
    refit="recall",
    cv=cv,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)

results = pd.DataFrame(grid_search.cv_results_)

best_row = results.loc[grid_search.best_index_]

print("\nCross-Validation Scores of Selected Configuration:")
for metric in ["accuracy", "precision", "recall", "f1", "roc_auc"]:
    print(f"{metric}: {best_row[f'mean_test_{metric}']:.4f}")

In [ ]:
# Retrieve the best model (5 selected features)
best_model = grid_search.best_estimator_

print("Best parameters:", grid_search.best_params_)
print("Selected model:", best_model)

# Evaluate the selected model using 10-fold CV
scores_selected_new = cross_validate(
    estimator=best_model,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring={
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1",
        "roc_auc": "roc_auc"
    },
    n_jobs=-1
)

# Create a fresh results DataFrame
results_selected_new = pd.DataFrame({
    metric: scores_selected_new[f"test_{metric}"]
    for metric in ["accuracy", "precision", "recall", "f1", "roc_auc"]
})

display(results_selected_new.round(4))

In [ ]:
best_pipeline = grid_search.best_estimator_

selector = best_pipeline.named_steps["selection"]

# Get selected feature names
selected_features = X_train.columns[selector.get_support()]

print("Best k:", grid_search.best_params_["selection__k"])
print("\nSelected Features:")

for i, feature in enumerate(selected_features, start=1):
    print(f"{i}. {feature}")

In [ ]:
comparison_new = pd.DataFrame({
    "30 Features": results_30.mean(),
    "10 Mean Features": results_10.mean(),
    "Selected 5 Features": results_selected_new.mean()
}).T

comparison_std = pd.DataFrame({
    "30 Features": results_30.std(),
    "10 Mean Features": results_10.std(),
    "Selected 5 Features": results_selected_new.std()
}).T

display(comparison_new.round(4))
print("Standard Deviation across folds:")
display(comparison_std.round(4))

In [ ]:
ranked = comparison_new.sort_values(
    by=["recall", "roc_auc"],
    ascending=False
)

best_model_name = ranked.index[0]

print("Selected model:", best_model_name)

# Choose the final model
if best_model_name == "30 Features":
    final_model = GaussianNB()
    final_X_train = X_train
    final_X_test = X_test

elif best_model_name == "10 Mean Features":
    final_model = GaussianNB()
    final_X_train = X_train_10
    final_X_test = X_test_10

else:
    final_model = grid_search.best_estimator_
    final_X_train = X_train
    final_X_test = X_test

# Train final model on all training data
final_model.fit(final_X_train, y_train)

print("Final model trained.")

In [ ]:
from scipy.stats import norm

selected_features = [
    "perimeter1",
    "concave_points1",
    "radius3",
    "perimeter3",
    "concave_points3"
]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, feature in enumerate(selected_features):
    ax = axes[i]

    for label, name, color in [
        (0, "Benign", "blue"),
        (1, "Malignant", "red")
    ]:
        values = X_train.loc[y_train == label, feature]

        # Actual distribution
        ax.hist(
            values,
            bins=20,
            density=True,
            alpha=0.25,
            color=color
        )

        # Fit Gaussian distribution
        mu = values.mean()
        sigma = values.std(ddof=0)

        x = np.linspace(values.min(), values.max(), 200)

        ax.plot(
            x,
            norm.pdf(x, mu, sigma),
            color=color,
            linewidth=2,
            label=f"{name}"
        )

    ax.set_title(feature)
    ax.set_xlabel("Values of each feature")
    ax.set_ylabel("Density")
    ax.legend()
    ax.grid(alpha=0.2)

# Remove unused subplot
fig.delaxes(axes[5])

plt.suptitle(
    "Gaussian Distributions of Selected Features by Diagnosis",
    fontsize=15
)

plt.tight_layout()
plt.show()

In [ ]:
y_pred = final_model.predict(final_X_test)
y_prob = final_model.predict_proba(final_X_test)[:, 1]

test_results = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1": f1_score(y_test, y_pred),
    "ROC-AUC": roc_auc_score(y_test, y_prob)
}

print("Final Test Performance:")
for metric, value in test_results.items():
    print(f"{metric}: {value:.4f}")

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=["Benign", "Malignant"],
    cmap="Blues",
    values_format="d"
)
plt.grid(False)
plt.title("Final Model - Test Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import roc_curve

# Calculate ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

# Calculate AUC
auc_score = roc_auc_score(y_test, y_prob)

# Plot ROC curve
plt.figure(figsize=(8, 6))

plt.plot(
    fpr,
    tpr,
    label=f"Gaussian Naive Bayes (AUC = {auc_score:.4f})",
    linewidth=2
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Recall)")
plt.title("ROC Curve - Final Gaussian Naive Bayes Model")

plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import cross_val_predict

# Pick a cutoff using out-of-fold probabilities on the TRAINING data only
oof_prob = cross_val_predict(grid_search.best_estimator_, X_train, y_train,
                             cv=cv, method="predict_proba")[:, 1]
for t in [0.5, 0.2, 0.1, 0.05]:
    tn, fp, fn, tp = confusion_matrix(y_train, (oof_prob >= t).astype(int)).ravel()
    print(f"Train CV, cutoff {t}: missed {fn}/{fn+tp}, false alarms {fp}/{fp+tn}")

# Check the chosen cutoff once on the test set
for t in [0.5, 0.1]:
    tn, fp, fn, tp = confusion_matrix(y_test, (y_prob >= t).astype(int)).ravel()
    print(f"Test, cutoff {t}: missed {fn}/{fn+tp}, false alarms {fp}/{fp+tn}")